In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd

## Parámetros del portafolio
En lugar de número de acciones, aquí definimos:
- `portfolio_value`: valor total del portafolio en dólares
- `weights_dict`: ponderación de cada activo (positivo = largo, negativo = corto). La suma de los valores absolutos debe ser 1.0

In [ ]:
portfolio_value = 1_000_000  # Valor total del portafolio en USD

# Ponderaciones: positivo = largo, negativo = corto
# La suma de los valores absolutos debe ser igual a 1.0
weights_dict = {
    'KO':  -0.125,
    'PEP':  0.125,
    'V':    0.125,
    'MA':  -0.125,
    'XOM': -0.125,
    'CVX':  0.125,
    'JPM':  0.125,
    'BAC': -0.125
}

# Validación: suma de valores absolutos debe ser 1
total_abs = sum(abs(w) for w in weights_dict.values())
print(f"Suma de |pesos|: {total_abs:.4f}  {'✓ OK' if abs(total_abs - 1.0) < 1e-6 else '✗ ERROR: no suman 1'}")

## Descarga de datos históricos

In [ ]:
tickers = list(weights_dict.keys())

data = yf.download(tickers, start='2024-02-12', end='2026-02-13', progress=False)['Close']
returns = data.pct_change().dropna()

print(f"Período: {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Observaciones: {len(returns)} días")

## Pesos como Series de pandas
Al construir `weights` como una Series con los mismos nombres de columna que `returns`, pandas alinea automáticamente por ticker al hacer el producto punto — sin importar el orden.

In [ ]:
weights = pd.Series(weights_dict)

pd.DataFrame(weights, columns=['Peso']).style.format({'Peso': '{:.1%}'})

## Retornos del portafolio y P&L histórico

In [ ]:
# Retorno diario del portafolio (escalar por día)
portfolio_returns = returns.dot(weights)

# P&L diario en dólares
portfolio_pnl = portfolio_returns * portfolio_value

print(f"Retorno diario promedio:  {portfolio_returns.mean():.4%}")
print(f"Volatilidad diaria:       {portfolio_returns.std():.4%}")

## VaR Histórico (Historical Simulation)

In [ ]:
VaR_95 = np.percentile(portfolio_pnl, 5)
VaR_99 = np.percentile(portfolio_pnl, 1)

print("=== VaR Histórico 1 día ===")
print(f"  VaR 95%: ${VaR_95:,.2f}")
print(f"  VaR 99%: ${VaR_99:,.2f}")

## Análisis de liquidez
Estimamos el número de acciones implícito a partir de los pesos y el precio actual, luego calculamos cuántos días tomaría liquidar cada posición operando máximo el 10% del volumen diario promedio.

In [ ]:
volume = yf.download(tickers, start='2025-11-12', end='2026-02-13', progress=False)['Volume']
avg_volume = volume.mean()

latest_prices = data.iloc[-1]

# Número implícito de acciones = (peso * valor_portafolio) / precio
implied_shares = (weights.abs() * portfolio_value) / latest_prices

liquidity = pd.DataFrame({
    'Precio':           latest_prices,
    'Acciones implícitas': implied_shares,
    'Vol. diario prom.':   avg_volume,
    'Days to Liquidate':   implied_shares / (0.10 * avg_volume)
})
liquidity['Cumple (<2 días)'] = liquidity['Days to Liquidate'] < 2

liquidity.style.format({
    'Precio': '${:.2f}',
    'Acciones implícitas': '{:,.1f}',
    'Vol. diario prom.': '{:,.0f}',
    'Days to Liquidate': '{:.4f}'
})